In [6]:
import requests 

url = "https://books.toscrape.com/index.html"
#Importation de Google.com
response = requests.get(url)

# Des premiers exemples de ce qu'il est possible de faire.

In [7]:
#Savoir si la requête q bien été effectuée (code 200 = C'est bon)

print(response.status_code)

200


In [25]:
#Récupérer tout le code HTML de la page

print(response.text)



<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:33" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="../static/oscar/fav

In [9]:
#Créer un fichier HTML composé du code HTML scrappé.

with open("index.html", "w") as f:
    f.write(response.text)

In [10]:
#Ouvrir le code du fichier html scrappé

with open("index.html","r") as f:
    #print(f.read())
    pass

# Découverte des bases de BS4

In [11]:
from bs4 import BeautifulSoup
from pprint import pprint

with open("index.html","r") as f:
    data = f.read()
    
#lxml-xml permet de parser du xml
#html.parser très rapide mais plus de risque d'erreur. -> conseillé si pas d'erreurs visibles.
#html5lib moins rapide mais moins de risques d'erreur.

soup = BeautifulSoup(data,features="html.parser")

In [24]:
pprint(soup.prettify()) #Affiche le html de manière structurée.

('<!DOCTYPE html>\n'
 '<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 '
 'lt-ie7"> <![endif]-->\n'
 '<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> '
 '<![endif]-->\n'
 '<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> '
 '<![endif]-->\n'
 '<!--[if gt IE 8]><!-->\n'
 '<html class="no-js" lang="en-us">\n'
 ' <!--<![endif]-->\n'
 ' <head>\n'
 '  <title>\n'
 '   All products | Books to Scrape - Sandbox\n'
 '  </title>\n'
 '  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>\n'
 '  <meta content="24th Jun 2016 09:33" name="created"/>\n'
 '  <meta content="" name="description"/>\n'
 '  <meta content="width=device-width" name="viewport"/>\n'
 '  <meta content="NOARCHIVE,NOCACHE" name="robots"/>\n'
 '  <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->\n'
 '  <!--[if lt IE 9]>\n'
 '        <script '
 'src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>\n'
 '        <![endif]-->\n'
 '  <link href=

## Les méthodes find et find_all:

In [13]:
#Récupérer tous les catégories du site scrappé:

aside = soup.find("div",class_="side_categories")
categories_div = aside.find("ul").find('li').find("ul")
categories = [category.text.strip() for category in categories_div.children if category.name]

print(categories)

['Travel', 'Mystery', 'Historical Fiction', 'Sequential Art', 'Classics', 'Philosophy', 'Romance', 'Womens Fiction', 'Fiction', 'Childrens', 'Religion', 'Nonfiction', 'Music', 'Default', 'Science Fiction', 'Sports and Games', 'Add a comment', 'Fantasy', 'New Adult', 'Young Adult', 'Science', 'Poetry', 'Paranormal', 'Art', 'Psychology', 'Autobiography', 'Parenting', 'Adult Fiction', 'Humor', 'Horror', 'History', 'Food and Drink', 'Christian Fiction', 'Business', 'Biography', 'Thriller', 'Contemporary', 'Spirituality', 'Academic', 'Self Help', 'Historical', 'Christian', 'Suspense', 'Short Stories', 'Novels', 'Health', 'Politics', 'Cultural', 'Erotica', 'Crime']


In [26]:
#Récupérer les images contenues dans le site.

images = soup.find("section").find_all("img")
src =  [l.get('src') for l in images] #Affiche le lien des premières de couverture des livres.
# print(src)

articles = soup.find_all("article",class_="product_pod")
title = [article.find_all("a")[-1].get('title') for article in articles]
print(title)

['Frankenstein', 'Forever Rockers (The Rocker #12)', 'Fighting Fate (Fighting #6)', 'Emma', 'Eat, Pray, Love', 'Deep Under (Walker Security #1)', "Choosing Our Religion: The Spiritual Lives of America's Nones", 'Charlie and the Chocolate Factory (Charlie Bucket #1)', "Charity's Cross (Charles Towne Belles #4)", 'Bright Lines', "Bridget Jones's Diary (Bridget Jones #1)", 'Bounty (Colorado Mountain #7)', 'Blood Defense (Samantha Brinkman #1)', 'Bleach, Vol. 1: Strawberry and the Soul Reapers (Bleach #1)', 'Beyond Good and Evil', "Alice in Wonderland (Alice's Adventures in Wonderland #1)", 'Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)', "A Spy's Devotion (The Regency Spies of London #1)", "1st to Die (Women's Murder Club #1)", '1,000 Places to See Before You Die']


## Exercice n°1: Rechercher tous les livres disponibles dans chaque catégorie comportant moins de 20 livres et les mettre dnas un dictionnaire PYTHON.

### Ma méthode:

In [16]:
#Récupérer le index.html de la page d'accueil du site
with open("index.html", "r") as f:
    data = f.read()

#Récupérer le nom des catégories ainsi que le lien envoyant vers la catégorie en question.
aside = soup.find("div",class_="side_categories")
categories_div = aside.find("ul").find('li').find("ul")
categories = [category.text.strip() for category in categories_div.children if category.name]
cat_links = [category.find("a").get("href") for category in categories_div.children if category.find("a") != -1]

In [17]:
#On crée un dictionnaire vide que l'on va agrémenter par la suite
dict_books_cats = {}
seuil = 19
#On itère sur tous les liens (et on récupère l'index pour ensuite pouvoir lui attribuer le nom de la catégorie correspondante.)
for index, link in enumerate(cat_links):
    
    #On lance une requête sur la catégorie actuelle pour pouvoir créer un objet BS4 relié.
    request = requests.get(f"https://books.toscrape.com/{link}")
    soup_cat = BeautifulSoup(request.text,features="html.parser")
    
    #On récupère le nom des livres présents sur la catégorie actuelle.
    articles_cat = soup_cat.find_all("article",class_="product_pod")
    title_cat = [article.find_all("a")[-1].get('title') for article in articles_cat]

    #On lie les noms des catégories aux livres présents dans la catégorie.
    if len(title_cat) <= seuil:
        dict_books_cats[f"{categories[index]}"] = title_cat
    

In [18]:
len(dict_books_cats.keys())

38

### La méthode de la formation suivie:

In [19]:
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/index.html"


def main(threshold=19):
  dict_books_cats = {}

  with requests.Session() as session:
    # 1. Page d'accueil pour récupérer les catégories et leurs liens d'un coup
    response = session.get(BASE_URL)
    soup = BeautifulSoup(response.text, "html.parser")

    # Sélection des liens de sous-catégories dans le menu latéral
    category_links = soup.select("div.side_categories ul.nav.nav-list ul li a")

    # 2. Itération propre sans gestion d'index fragile
    for cat_el in category_links:
      category_name = cat_el.text.strip()
      link = cat_el.get("href")
      absolute_url = urljoin(BASE_URL, link)

      # 3. Requête via la session
      resp = session.get(absolute_url)
      soup_cat = BeautifulSoup(resp.text, "html.parser")

      # 4. Sélection via CSS (équivalent du find_all / find[-1])
      articles_cat = soup_cat.select("article.product_pod")
      title_cat = [
          art.select("a")[-1].get("title")
          for art in articles_cat
          if art.select("a")
      ]

      # 5. Filtrage et remplissage
      if len(title_cat) <= threshold:
        dict_books_cats[category_name] = title_cat

  return dict_books_cats

In [20]:
test = main()

In [21]:
len(test.keys())

38

### Exercice 2: Récupérer dans un dictionnaire Python tous les livres notés 1 étoile ainsi que leur id.

In [22]:
import re
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

START_URL = "https://books.toscrape.com/index.html"


def map_rank_number(classes):
  if not classes:
    return "unknown"
  rating_map = {
      "one": "1-star",
      "two": "2-star",
      "three": "3-star",
      "four": "4-star",
      "five": "5-star",
  }
  for c in classes:
    if c.lower() in rating_map:
      return rating_map[c.lower()]
  return "unknown"


with requests.Session() as session:
  dict_rank = {f"{i+1}-star": {} for i in range(5)}
  dict_rank["unknown"] = {}

  current_url = START_URL
  first_page = True

  while current_url:
    response = session.get(current_url)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.select("article.product_pod")
    if not books:
      break

    for book in books:
      # Nom du produit
      img_tag = book.select_one("img")
      name = img_tag.get("alt", "Inconnu") if img_tag else "Inconnu"

      # ID du produit
      a_tag = book.select_one("h3 a")
      book_id = "0"
      if a_tag and a_tag.get("href"):
        match = re.search(r"(?<=_)\d+", a_tag.get("href"))
        if match:
          book_id = match.group()

      # Note d'étoile
      p_star = book.select_one("p.star-rating")
      classes = p_star.get("class", []) if p_star else []
      rank = map_rank_number(classes)

      if rank in dict_rank:
        dict_rank[rank][book_id] = [name, first_page]

    first_page = False

    # Sécurité pour la pagination
    next_link_el = soup.select_one("li.next a")
    if next_link_el and next_link_el.get("href"):
      current_url = urljoin(current_url, next_link_el.get("href"))
    else:
      current_url = None  # plus de page suivante

print(
    f"Terminé ! 1-star: {len(dict_rank['1-star'])} livres enregistrés"
)

Terminé ! 1-star: 226 livres enregistrés
